# PSM Memory LOCOMO ingestion on Colab

This notebook is for a fresh LOCOMO ingestion smoke run on Colab. It clones the PSM repo, builds the local workspace packages, installs those local package tarballs, prepares the model with `psm-memory setup`, ingests a small batch into SQLite, and lets you download or sync the DB.

Default mode is a fresh 20-turn smoke run. Set `RESUME_FROM_HF = True` only when intentionally continuing a previous run from Hugging Face checkpoints.

Recommended Colab runtime: GPU. Keep `LIMIT = 20` for smoke validation before attempting larger runs.

In [ ]:
# Install Node.js 22. Colab's default Node can be older than the package engine range.
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Keep PSM caches under /content.
# Hugging Face Hub is the portable checkpoint store across Colab accounts.
# Build PSM from the cloned repo and install local package tarballs instead of npm latest.
from pathlib import Path
import json
import os
import shutil

HF_REPO_ID = "chkrishna2001/psm-memory-locomo-artifacts"  # Change to your private HF dataset repo if needed.
HF_LOCAL_DIR = "/content/locomo/hf-artifacts"
WORK_DIR = "/content/psm-memory-work"
REPO_DIR = "/content/PSM"
RESULT_DIR = "/content/locomo/results"
DATA = "/content/PSM/benchmark/locomo/data/locomo10.json"
MODEL = "/content/psm-memory-cache/psm-memory-qwen-1.5b-q4_k_m.gguf"
RUN_LABEL = "smoke-20"
DB = f"/content/locomo/results/locomo-psm-memory-{RUN_LABEL}.db"
CHECKPOINT_DIR = f"/content/locomo/checkpoints/{RUN_LABEL}"
PROGRESS = f"/content/locomo/results/locomo-ingest-progress-{RUN_LABEL}.json"
LIMIT = 20
BATCH_SIZE = 5
WINDOW_SIZE = 2
CONTEXT_SIZE = 4096
RESUME_FROM_HF = False

%env PSM_MEMORY_SKIP_SETUP=1
%env PSM_MEMORY_HOME=/content/psm-memory-cache
!mkdir -p "$WORK_DIR" "$RESULT_DIR" "$CHECKPOINT_DIR" "$HF_LOCAL_DIR"
!rm -rf "$REPO_DIR"
!git clone --depth 1 https://github.com/chkrishna2001/PSM.git "$REPO_DIR"
%cd /content/PSM
!npm ci
!npm run build
!rm -f "$WORK_DIR"/psm-memory-*.tgz
!npm pack -w @psm-memory/sdk --pack-destination "$WORK_DIR"
!npm pack -w @psm-memory/cli --pack-destination "$WORK_DIR"
%cd /content/psm-memory-work
!npm init -y >/dev/null 2>&1 || true
!npm install ./psm-memory-sdk-*.tgz ./psm-memory-cli-*.tgz
!cp /content/PSM/benchmark/locomo/colab/locomo-benchmark.mjs /content/psm-memory-work/locomo-benchmark.mjs
!ls -lh /content/PSM/benchmark/locomo/data/locomo10.json /content/psm-memory-work/locomo-benchmark.mjs
!node -e "console.log(require('fs').existsSync('/content/psm-memory-work/node_modules/@psm-memory/sdk/dist/index.js') ? 'local sdk installed' : 'missing sdk')"

In [ ]:
# Configure Hugging Face Hub artifact sync. This is portable across Colab Google accounts.
# Create a private dataset repo if it does not exist, then restore any prior checkpoints.
!pip -q install -U huggingface_hub
import getpass
import os
import shutil
import subprocess
import time
from pathlib import Path

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN') or (userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_TOKEN') or '')
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

os.environ['HF_REPO_ID'] = HF_REPO_ID
os.environ['HF_LOCAL_DIR'] = HF_LOCAL_DIR
!huggingface-cli repo create "$HF_REPO_ID" --type dataset --private -y >/dev/null 2>&1 || true

def hf_download(path_in_repo, local_dir):
    cmd = [
        'huggingface-cli', 'download', HF_REPO_ID,
        '--repo-type', 'dataset',
        '--local-dir', local_dir,
        path_in_repo,
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode == 0:
        print(f"Restored hf://{HF_REPO_ID}/{path_in_repo}")
        return True
    print(f"No HF artifact yet: {path_in_repo}")
    return False

Path(HF_LOCAL_DIR).mkdir(parents=True, exist_ok=True)
if RESUME_FROM_HF:
    hf_download(f'ingest/checkpoints/{RUN_LABEL}/{Path(DB).name}', HF_LOCAL_DIR)
    hf_download(f'ingest/checkpoints/{RUN_LABEL}/{Path(DB).name}-wal', HF_LOCAL_DIR)
    hf_download(f'ingest/checkpoints/{RUN_LABEL}/{Path(DB).name}-shm', HF_LOCAL_DIR)
    hf_download(f'ingest/locomo-ingest-progress-{RUN_LABEL}.json', HF_LOCAL_DIR)
else:
    print('Fresh smoke run: skipping HF checkpoint restore.')

hf_checkpoint_dir = Path(HF_LOCAL_DIR) / 'ingest' / 'checkpoints' / RUN_LABEL
if RESUME_FROM_HF and (hf_checkpoint_dir / Path(DB).name).exists():
    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy2(hf_checkpoint_dir / Path(DB).name, Path(CHECKPOINT_DIR) / Path(DB).name)
    for suffix in ('-wal', '-shm'):
        sidecar = hf_checkpoint_dir / f"{Path(DB).name}{suffix}"
        if sidecar.exists():
            shutil.copy2(sidecar, Path(CHECKPOINT_DIR) / sidecar.name)
if RESUME_FROM_HF and (Path(HF_LOCAL_DIR) / 'ingest' / f'locomo-ingest-progress-{RUN_LABEL}.json').exists():
    shutil.copy2(Path(HF_LOCAL_DIR) / 'ingest' / f'locomo-ingest-progress-{RUN_LABEL}.json', PROGRESS)

sync_script = Path('/content/hf_sync_ingest.sh')
sync_script.write_text(f'''#!/usr/bin/env bash
set +e
while true; do
  mkdir -p "{HF_LOCAL_DIR}/ingest/checkpoints/{RUN_LABEL}" "{HF_LOCAL_DIR}/ingest/results"
  cp -f "{PROGRESS}" "{HF_LOCAL_DIR}/ingest/locomo-ingest-progress-{RUN_LABEL}.json" 2>/dev/null || true
  cp -f "{CHECKPOINT_DIR}"/* "{HF_LOCAL_DIR}/ingest/checkpoints/{RUN_LABEL}/" 2>/dev/null || true
  cp -f "{RESULT_DIR}"/* "{HF_LOCAL_DIR}/ingest/results/" 2>/dev/null || true
  huggingface-cli upload "{HF_REPO_ID}" "{HF_LOCAL_DIR}/ingest" ingest --repo-type dataset --commit-message "sync ingest artifacts" >/tmp/hf-sync-ingest.log 2>&1 || true
  sleep 60
done
''')
sync_script.chmod(0o755)
!pkill -f hf_sync_ingest.sh >/dev/null 2>&1 || true
!nohup bash /content/hf_sync_ingest.sh >/tmp/hf-sync-ingest.nohup 2>&1 &
print(f"HF background sync started for dataset repo: {HF_REPO_ID}")

In [ ]:
# Restore the latest checkpointed DB from Hugging Face only when RESUME_FROM_HF is true.
checkpoint_db = Path(CHECKPOINT_DIR) / Path(DB).name
if RESUME_FROM_HF and checkpoint_db.exists():
    Path(DB).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(checkpoint_db, DB)
    for suffix in ("-wal", "-shm"):
        sidecar = Path(str(checkpoint_db) + suffix)
        if sidecar.exists():
            shutil.copy2(sidecar, str(DB) + suffix)
    print(f"Restored DB checkpoint from {checkpoint_db}")
elif RESUME_FROM_HF:
    print("No DB checkpoint found in HF yet.")
else:
    for path in [DB, str(DB) + '-wal', str(DB) + '-shm', PROGRESS]:
        if Path(path).exists():
            Path(path).unlink()
    print(f"Fresh smoke run will write {DB}")

if RESUME_FROM_HF and Path(PROGRESS).exists():
    progress = json.loads(Path(PROGRESS).read_text())
    print(f"Progress file found: next_index={progress.get('next_index', 0)}")
else:
    print("No progress file found yet.")

In [ ]:
# Prepare the default model once using the locally built CLI. Embeddings are skipped for ingestion-only runs.
!./node_modules/.bin/psm-memory setup --skip-embeddings --yes --pretty

In [ ]:
# Fresh smoke ingest by default. Every batch writes local DB/progress checkpoints; the background HF sync uploads them.
!node /content/psm-memory-work/locomo-benchmark.mjs ingest --data "$DATA" --db "$DB" --model "$MODEL" --limit $LIMIT --batch-size $BATCH_SIZE --window-size $WINDOW_SIZE --context-size $CONTEXT_SIZE --progress "$PROGRESS" --checkpoint-dir "$CHECKPOINT_DIR"
!mkdir -p "$HF_LOCAL_DIR/ingest/checkpoints/$RUN_LABEL" "$HF_LOCAL_DIR/ingest/results"; cp -f "$PROGRESS" "$HF_LOCAL_DIR/ingest/locomo-ingest-progress-$RUN_LABEL.json" 2>/dev/null || true; cp -f "$CHECKPOINT_DIR"/* "$HF_LOCAL_DIR/ingest/checkpoints/$RUN_LABEL/" 2>/dev/null || true; cp -f "$RESULT_DIR"/* "$HF_LOCAL_DIR/ingest/results/" 2>/dev/null || true; huggingface-cli upload "$HF_REPO_ID" "$HF_LOCAL_DIR/ingest" ingest --repo-type dataset --commit-message "final ingest artifacts $RUN_LABEL" || true


In [ ]:
# Inspect local and Hugging Face sync artifacts.
!ls -lh /content/locomo/results
!ls -lh "$CHECKPOINT_DIR"
!find "$HF_LOCAL_DIR" -maxdepth 3 -type f -printf '%p %k KB\n' 2>/dev/null | sort | tail -50
summary = Path('/content/locomo/results/ingest-summary.json')
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2)[:4000])

In [ ]:
# Download the smoke DB and summary to your machine.
from google.colab import files
for path in [DB, '/content/locomo/results/ingest-summary.json', PROGRESS]:
    if Path(path).exists():
        print(f'Downloading {path}')
        files.download(path)
    else:
        print(f'Missing {path}')


If you want one checkpoint per turn, keep `BATCH_SIZE = 1`. For faster but less granular recovery, raise it to a larger batch size. The resume point is tracked in `ingest/locomo-ingest-progress.json` and the SQLite checkpoint lives under `ingest/checkpoints/` in the Hugging Face dataset repo.